In [ ]:
import math
from pathlib import Path

from matplotlib.dates import DateFormatter
import matplotlib.pyplot as plt
import polars as pl

from constants import VALIDATION_WINDOWS

## Naive Baseline Model

In [ ]:
class NaiveBaseline:
    def __init__(self, target_col: str, timestamp_col: str = "timestamp", period: int = 1):
        self.target_col = target_col
        self.timestamp_col = timestamp_col
        self.period = period
        self._last_period_df: pl.DataFrame | None = None

    @property
    def is_fit(self) -> bool:
        return self._last_period_df is not None
    
    def fit(self, X: pl.DataFrame) -> "NaiveBaseline":
        self._last_period_df = X.slice(offset=-self.period)
        return self

    def predict(self, n_steps: int = 1) -> pl.Series:
        assert self.is_fit
        n_repeats = math.ceil(n_steps / self.period)
        y_hat = pl.concat(items=[self._last_period_df for _ in range(n_repeats)])
        y_hat = y_hat.select(pl.col(self.target_col)).slice(0, n_steps)
        return y_hat.to_series()
        

## PJM Dataset

In [ ]:
PJM_SITE_NAME = "PJMW"
PJM_DATA_FREQUENCY = "1h"

INPUT_PATH = Path("../../data/pjm")
OUTPUT_PATH = Path(f"../../results/pjm/naive/{PJM_SITE_NAME}")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

data_file_name = f"{PJM_SITE_NAME}_hourly_processed.pq"
data_file_path = INPUT_PATH / data_file_name
SITE_DF = pl.read_parquet(data_file_path).sort(by="timestamp")

### Configure

In [ ]:
SEASONAL_PERIOD = int(24 * 7)
TARGET_COL = f"{PJM_SITE_NAME}_MW"

In [ ]:
for val_idx, (val_start, val_end) in enumerate(VALIDATION_WINDOWS[PJM_SITE_NAME]):
    train_df = SITE_DF.filter(pl.col("timestamp").lt(val_start))
    val_df = SITE_DF.filter(pl.col("timestamp").is_between(val_start, val_end, closed="left"))
    
    model = NaiveBaseline(target_col=TARGET_COL, period=SEASONAL_PERIOD)
    model = model.fit(train_df)
    y_hat = model.predict(n_steps=len(val_df)).rename(f"{TARGET_COL}_FORECAST")
    forecast_df = val_df.with_columns(y_hat)

    # Save forecasts
    forecast_output_path = f"{OUTPUT_PATH}/forecasts_{PJM_SITE_NAME}_fold_{val_idx}.pq"
    forecast_df.to_pandas().to_parquet(forecast_output_path)
    
    # Plot forecasts
    fig, ax = plt.subplots()
    
    ax.plot(forecast_df["timestamp"], forecast_df[f"{PJM_SITE_NAME}_MW"], color="black", lw=2, label="Actual")
    ax.plot(forecast_df["timestamp"], forecast_df[f"{PJM_SITE_NAME}_MW_FORECAST"], color="#0072B2", lw=2, label="Forecast")
    
    ax.legend(loc=1)
    ax.grid(True, which="major", c="grey", ls="--", lw=1, alpha=0.2)
    ax.set(ylabel="Load (kWh)", title=f"Electricity Load Forecasts for Site {PJM_SITE_NAME} (Fold {val_idx})")
    
    ax.xaxis.set_major_formatter(DateFormatter("%Y-%m-%d"))
    ax.tick_params(axis='x', labelrotation=45)
    
    fig.tight_layout()
    plt.savefig(f"{OUTPUT_PATH}/forecasts_{PJM_SITE_NAME}_fold_{val_idx}.png", dpi=300);
    plt.close(fig);